# 등급분류 개선 실험 — Baseline vs 개선 모델 비교 (Colab)

교수님 자문 반영: **"순수 모델 vs 우리가 개선한 모델을 정확히 비교해 기여(contribution)를 실증하라."**

| 실험 | 내용 | 가설 |
|---|---|---|
| **Baseline** | YOLO11m-cls, v2 데이터 그대로 | 기준선 (소스 단위 분할 = 누수 없음) |
| **실험 A** | 전처리: 흑백 + CLAHE 밝기 정규화 | 촬영 조도·색 편차 통제 → 일반화 ↑ |
| **실험 C** | 클래스 균형: 우수/보통 오버샘플링 | '보통' 재현율 개선 (불량 편중 완화) |

- 데이터: `house_grade_cls_v2.zip` (**촬영 소스 단위 분할** — train/val 소스 겹침 0)
- 평가: 등급 정확도 + **불량 재현율 / 위험누락률** (안전 핵심 지표)

**런타임 → GPU(A100/L4)** 설정 후 위에서부터 실행. (3회 학습이라 총 1~2시간)

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. 드라이브 연결 + v2 데이터 압축해제
`house_grade_cls_v2.zip` 이 My Drive 최상위에 있다고 가정.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, glob
ZIP_PATH = '/content/drive/MyDrive/house_grade_cls_v2.zip'
assert os.path.exists(ZIP_PATH), f'파일 없음: {ZIP_PATH}'
!rm -rf /content/house_grade_cls_v2
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content')

DATA = '/content/house_grade_cls_v2'
assert os.path.isdir(f'{DATA}/train'), '압축 구조 확인 필요'
for split in ('train', 'val'):
    print(split, {os.path.basename(d): len(os.listdir(d)) for d in sorted(glob.glob(f'{DATA}/{split}/*'))})

## 3. 공통 준비 — 평가 함수 (혼동행렬 + 핵심 지표)

In [ ]:
%pip install -q ultralytics opencv-python-headless
import glob, os, cv2, shutil, random
from collections import defaultdict
from ultralytics import YOLO

KO = {'good': '우수', 'fair': '보통', 'poor': '불량'}
GROUPS = ['우수', '보통', '불량']
results_table = {}

def evaluate(model, val_dir, name, preprocess=None):
    """val 폴더 평가 → 혼동행렬 출력 + (정확도, 불량재현율, 위험누락률) 저장."""
    cm = defaultdict(int); n = 0
    for cls_dir in glob.glob(f'{val_dir}/*'):
        tg = KO[os.path.basename(cls_dir)]
        for img in glob.glob(f'{cls_dir}/*'):
            src = preprocess(img) if preprocess else img
            r = model.predict(src, imgsz=320, verbose=False)[0]
            pg = KO[model.names[int(r.probs.top1)]]
            cm[(tg, pg)] += 1; n += 1
    acc = sum(cm[(g, g)] for g in GROUPS) / n
    dtot = sum(cm[('불량', p)] for p in GROUPS)
    tp = cm[('불량', '불량')]
    fair_tot = sum(cm[('보통', p)] for p in GROUPS)
    fair_rec = cm[('보통', '보통')] / fair_tot if fair_tot else 0
    print(f'\n===== {name} (val {n}장) =====')
    print('실제\\예측 |', ' | '.join(f'{g:>4}' for g in GROUPS))
    for t in GROUPS:
        print(f'{t:>6}   |', ' | '.join(f'{cm[(t,p)]:>4}' for p in GROUPS))
    print(f'정확도 {acc:.1%} | 불량 재현율 {tp/dtot:.1%} | 위험누락률 {(dtot-tp)/dtot:.1%} | 보통 재현율 {fair_rec:.1%}')
    results_table[name] = (acc, tp/dtot, (dtot-tp)/dtot, fair_rec)
    return cm

## 4. Baseline — v2 그대로 학습

In [ ]:
base = YOLO('yolo11m-cls.pt')
base_res = base.train(data=DATA, epochs=50, imgsz=320, batch=64, patience=12, name='grade_base_v2')
base_best = YOLO(os.path.join(base_res.save_dir, 'weights', 'best.pt'))
evaluate(base_best, f'{DATA}/val', 'Baseline(v2)')

## 5. 실험 A — 전처리 (흑백 + CLAHE 밝기 정규화)
촬영 조도·색 편차를 통제해 "어디서 찍든 같은 조건"으로. (교수님 조언 반영)
> 서비스 적용 시 서버 추론에도 동일 전처리 필요.

In [ ]:
def prep_img(src, dst):
    img = cv2.imread(src)
    if img is None: return False
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    eq = clahe.apply(gray)
    cv2.imwrite(dst, cv2.cvtColor(eq, cv2.COLOR_GRAY2BGR))  # 3채널 유지
    return True

PREP = '/content/house_grade_prepA'
if not os.path.isdir(PREP):
    for split in ('train', 'val'):
        for cls_dir in glob.glob(f'{DATA}/{split}/*'):
            out_d = f'{PREP}/{split}/{os.path.basename(cls_dir)}'
            os.makedirs(out_d, exist_ok=True)
            for img in glob.glob(f'{cls_dir}/*'):
                prep_img(img, os.path.join(out_d, os.path.basename(img)))
print('전처리 완료:', sum(len(files) for _, _, files in os.walk(PREP)), '장')

expA = YOLO('yolo11m-cls.pt')
expA_res = expA.train(data=PREP, epochs=50, imgsz=320, batch=64, patience=12, name='grade_prepA')
expA_best = YOLO(os.path.join(expA_res.save_dir, 'weights', 'best.pt'))
evaluate(expA_best, f'{PREP}/val', '실험A(전처리)')

## 6. 실험 C — 클래스 균형 (우수/보통 오버샘플링)
train의 우수·보통을 불량 수준까지 복제 → '보통' 재현율 개선 기대. (val은 그대로)

In [ ]:
BAL = '/content/house_grade_balC'
if not os.path.isdir(BAL):
    shutil.copytree(f'{DATA}/val', f'{BAL}/val')
    counts = {}
    for cls_dir in glob.glob(f'{DATA}/train/*'):
        counts[os.path.basename(cls_dir)] = len(os.listdir(cls_dir))
    target = max(counts.values())
    random.seed(0)
    for cls_dir in glob.glob(f'{DATA}/train/*'):
        c = os.path.basename(cls_dir)
        out_d = f'{BAL}/train/{c}'
        os.makedirs(out_d, exist_ok=True)
        files = glob.glob(f'{cls_dir}/*')
        for f in files:
            shutil.copy(f, out_d)
        k = 0
        while len(os.listdir(out_d)) < target:  # 부족분 복제
            f = files[k % len(files)]; k += 1
            shutil.copy(f, os.path.join(out_d, f'dup{k}_' + os.path.basename(f)))
for split in ('train', 'val'):
    print(split, {os.path.basename(d): len(os.listdir(d)) for d in sorted(glob.glob(f'{BAL}/{split}/*'))})

expC = YOLO('yolo11m-cls.pt')
expC_res = expC.train(data=BAL, epochs=50, imgsz=320, batch=64, patience=12, name='grade_balC')
expC_best = YOLO(os.path.join(expC_res.save_dir, 'weights', 'best.pt'))
evaluate(expC_best, f'{BAL}/val', '실험C(클래스균형)')

## 7. ★ 최종 비교표 — 발표용 (Baseline vs 개선)

In [ ]:
print(f"{'모델':<16}{'정확도':>8}{'불량재현율':>10}{'위험누락률':>10}{'보통재현율':>10}")
for name, (acc, rec, miss, fair) in results_table.items():
    print(f'{name:<16}{acc:>7.1%}{rec:>9.1%}{miss:>9.1%}{fair:>9.1%}')
print('\n※ 안전 도구 핵심 = 위험누락률(낮을수록)·불량 재현율(높을수록). 보통 재현율은 실험 C 효과 확인용.')

## 8. 모델 저장 (드라이브 백업 + 다운로드)

In [ ]:
import shutil
from google.colab import files
for res, tag in ((base_res, 'base_v2'), (expA_res, 'prepA'), (expC_res, 'balC')):
    src = os.path.join(res.save_dir, 'weights', 'best.pt')
    dst = f'/content/drive/MyDrive/grade_{tag}_best.pt'
    shutil.copy(src, dst)
    print('드라이브 저장:', dst)
# 성능 가장 좋은 모델 하나를 로컬로 다운로드해 서버(grade_cls.pt)에 교체
files.download(os.path.join(base_res.save_dir, 'weights', 'best.pt'))